# Analisi dati per effetto Hall

A grandi linee:
1. caratterizzare l'**uniformità del campo magnetico** nel traferro
2. caratterizzare l'andamento del **campo magnetico** nel traferro al **variare della corrente** in ingresso
3. caratterizzare **tensione di Hall al variare della corrente** $V_H(i)$ (ferromagnete spento + 5x2 valori del campo magnetico)
4. caratterizzare **tensione di Hall al variare del campo magnetico** $V_H(B)$ (ferromagnete spento + 3x2 valori di corrente)
5. valutare **mobilità** dei portatori di carica nel materiale

+ cose facolative (?)

### utils

In [2]:
import numpy as np
from utils import meanCalc, fitPlotter, testZ, MeanError, Zscore
import ROOT

## Uniformità campo magnetico (TOTHINKABOUT)

Facendo misure del campo magnetico con una sonda di Hall vogliamo studiare l'uniformità di $B$ all'interno del traferro del ferromagnete.

In generale, la semidifferenza tra il valore massimo e il valore minimo del campo magnetico (all'interno della regione di omogeneità) sarà il limite minimo dell'errore su tutte le misure di campo magnetico.

In [3]:
# arrays of distances
posx = np.arange(5)*10
posy = np.arange(5)*10

# matrix of B values in mT
Bu = np.array([[176.20,204.5,202.0,203.6,164.16],
               [205.98,234.51,233.37,234.15,211.61],
               [215.00,236.,236.47,236.65,209.43],
               [204.67,235.5,235.5,235.3,207.4],
               [176.24,200.,201.10,202.05,175.02]])

Can2d_u = ROOT.TCanvas("c2d", "Uniformità campo magnetico")
H2d_u = ROOT.TH2F("h2d_u", "Uniformità campo magnetico; x [mm]; y [mm]; B [T]" ,5,0,5, 5,0,5)

for i in range(5):
    for j in range(5):
        H2d_u.SetBinContent(i+1,j+1,float(Bu[i][j]))

H2d_u.Draw("LEGO2")
Can2d_u.Draw()

ERROR! Session/line number was not unique in database. History logging moved to new session 5


Dopo lo studio di uniformità del campo magnetico, si stabilisce in quale zona effettuare le misure di campo magnetico. In tale zona, si prende la semidispersione di B come stima della risoluzione del campo magnetico.

In [4]:
B_max = 236.65
B_min = 233.4
resB = (B_max - B_min)/2
print(resB)

1.625


## Campo magnetico prodotto al variare della corrente

Tralasciando la prima curva di salita (da $0$ a $i_\text{max}$) misuriamo due salite e due discese complete (da $i_\text{max}$ a $-i_\text{max}$, e viceversa) prendendo 10 dati per ogni curva. Sostanzialmente, vogliamo verificare la regione di linearità in cui (dopo) vogliamo svolgere il resto dell'esperienza.

Escluse le zone di saturazione vogliamo un fare un fit lineare su ogni curva: per ognuna delle due coppie (due curve di salita, e due curve di discesa) stimiamo coi valori medi di $m$ e $q$ il vero coefficiente angolare e la vera quota (*l'errore sulla quota lo stimiamo con la semidifferenza tra i due valori*).
Solo dopo mediamo i risultati ottenuti per le curve di salita e quelle di discesa (*errore sulla quota sempre dato dalla semidifferenza*).

In [23]:
# we would like to see the hystheresis loop
# the error on i should be ( 0.5% rdg + 10 dgts ) what about the error on B?
# first negative run

B1 = [355.2, 322.5, 279.3, 243, 200.4, 159, 104.5, 59.8, 9.3, -47.7, -87.7, -136.4, -183.3, -228.5, -269.9, -314.1, -349.4, -383.8] #mT
i1 = [1.602, 1.406, 1.18, 0.999, 0.8, 0.619, 0.393, 0.208, 0, -0.234, -0.4, -0.603, -0.802, -1.011, -1.201, -1.419, -1.604, -1.805] #A
errB1 = [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
erri1 = [0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001] 

# first positive run
B2 = [-358.1, -322.4, -282.9, -244.4, -201.6, -154.4, -94.7, -58.9, -8.7, 40.2, 89.6, 138.9, 189.1, 231.9, 268.8, 310.6, 350.7, 381.7] #mT
i2 = [-1.605, -1.4, -1.195, -1.007, -0.806, -0.599, -0.353, -0.206, 0, 0.202, 0.405, 0.611, 0.825, 1.025, 1.198, 1.403, 1.615, 1.798] #A
errB2 = [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
erri2 = [0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001] 

# second negative run
B3 = [356.3, 319.6, 281.7, 243.8, 198, 157.7, 108.3, 56.7, 8.9, -44.5, -88.4, -141.8, -185.5, -227.3, -272.3, -313.6, -349.9, -383.8] #A
i3 = [1.599, 1.383, 1.192, 1.003, 0.788, 0.611, 0.409, 0.197, 0, -0.219, -0.4, -0.621, -0.808, -1.001, -1.213, -1.416, -1.606, -1.807] #mT
errB3 = [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
erri3 = [0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001] 

# second positive run 
# we have five more data as we wanted to see the magnetic saturation
B4 = [-358.7, -325.1, -284.8, -240.8, -202.5, -156, -106.2, -56.4, -8.7, 42.5, 87.2, 137.5, 190.2, 227.7, 272.7, 321.9, 349.5, 381.7, 414.9, 452, 476.7, 495.4, 499.9] #A
i4 = [-1.61, -1.409, -1.201, -0.987, -0.808, -0.603, -0.399, -0.195, 0, 0.21, 0.395, 0.602, 0.83, 1, 1.208, 1.456, 1.602, 1.798, 2.013, 2.318, 2.627, 2.9, 2.977] #mT
errB4 = [0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1]
erri4 = [0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001] 

magCurrPlotter = fitPlotter("MagneticFieldVsCurrent")
param1 = magCurrPlotter.addGraph(B1, i1, errB1, erri1, title="run1")
param2 = magCurrPlotter.addGraph(B2, i2, errB2, erri2, title="run2")
param3 = magCurrPlotter.addGraph(B3, i3, errB3, erri3, title="run3")
param4 = magCurrPlotter.addGraph(B4, i4, errB4, erri4, title="run4")
magCurrPlotter.drawCanvas()
magCurrPlotter.saveCanvas("outputs/MagneticFieldVsCurrent.png")


--- fit Results for: run1 ---
Function: pol1
Chi2/NDF: 20722.1069 / 16
p-value:  0.0000

p0: -0.0383 +/- 0.0003
p1: 0.0044 +/- 0.0000
--------------------------------

--- fit Results for: run2 ---
Function: pol1
Chi2/NDF: 20091.8268 / 16
p-value:  0.0000

p0: 0.0386 +/- 0.0003
p1: 0.0044 +/- 0.0000
--------------------------------

--- fit Results for: run3 ---
Function: pol1
Chi2/NDF: 21038.7400 / 16
p-value:  0.0000

p0: -0.0384 +/- 0.0003
p1: 0.0044 +/- 0.0000
--------------------------------

--- fit Results for: run4 ---
Function: pol1
Chi2/NDF: 586642.2261 / 21
p-value:  0.0000

p0: 0.0846 +/- 0.0003
p1: 0.0049 +/- 0.0000
--------------------------------


Warning in <TCanvas::Constructor>: Deleting canvas with same name: MagneticFieldVsCurrent
python ERROR: cannot open image file "outputs/MagneticFieldVsCurrent.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/MagneticFieldVsCurrent.png


NB: per i parametri ritornati dai fit, sto utilizzando il formato `np.array([[mean][error],...])` per scrivere i valori, così poi da avere le funzioni Zscore e MeanErrors per calcolare direttamente valori attessi ed errori

In [ ]:
# verifico che i parametri in salita e discesa siano compatibili, poi medio tutto

Zscore(param1, param3)
param_up = MeanError(param1, param3)

Zscore(param2, param4)
param_down = MeanError(param2, param4)

Zscore(param_up, param_down)
param = MeanError(param_up,param_down)

param

## Tensione di Hall al variare di $i$

Vogliamo valutare l'andamento della tensione di Hall (sui lati del materiale semiconduttore) al variare della corrente iniettata al suo interno, in modo da stimare il parametro $R_H$.

Quindi fissato il valore del campo magnetico (una volta a zero per eliminare il fondo, e poi a 5 valori diversi in entrambi i versi) valutiamo l'andamento $V_H(i)$ (*andando tra **-8mA e +8mA***) sapendo che in generale:
$$V_H = \frac{R_H}{t} \cdot i_p \cdot B$$
allora facciamo dei fit lineari su ogni set di dati e diamo una stima del parametro $R_H$ per entrambi i versi del campo magnetico, poi mediamo tra i due set.

In [ ]:
# correction for misalignment of transverse contacts
B0    = 0
errB0 = 0.1

VH0    = [-0.001, 0.055, 0.109, 0.165, 0.22, 0.276, 0.328, 0.384, 0.439, -0.051, -0.104, -0.159, -0.214, -0.267, -0.319, -0.374, -0.429]
errVH0 = [0.1]*17
Ip0    = [0, 1.009, 1.999, 2.989, 4, 5.012, 5.977, 6.977, 7.997, -0.996, -1.986, -2.998, -4.003, -4.995, -5.975, -6.985, -8.009]
errIp0 = [0.2]*17

hallCurrPlotter = fitPlotter("HallTensionVsCurrent")
param0 = hallCurrPlotter.addGraph(Ip0, VH0, errIp0, errVH0, "Ohmic behaviour")
hallCurrPlotter.drawCanvas()
hallCurrPlotter.saveCanvas("outputs/HallTensionVsCurrent.png")

Stima_VH0 = param0[1]
Stima_VH0

In [26]:
# measuring Vh while changing the current ( B = constant )
# remember to implement errors
# the error on i should be ( 0.5% rdg + 10 dgts ) what about the error of Vh?

i1 = np.array([0, -0.995, -1.973, -2.989, -3.994, -4.988, -5.972, -7.002, -7.985, 1.001, 2.001, 3.019, 4, 4.993, 5.989, 7.002, 8.014]) #mA
VH1 = np.array([-0.01, -0.479, -0.938, -1.405, -1.875, -2.341, -2.801, -3.284, -3.743, 0.461, 0.93, 1.406, 1.866, 2.331, 2.798, 3.273, 3.747]) #mV

i2 = np.array([0, 0.988, 1.996, 2.989, 4, 4.998, 6.004, 6.959, 8.012, -1.011, -1.968, -3.005, -4.005, -5.052, -5.98, -7.033, -8.004]) #mA
VH2 = np.array([-0.008, 0.847, 1.721, 2.58, 3.455, 4.319, 5.189, 6.016, 6.927, -0.885, -1.714, -2.612, -3.477, -4.383, -5.185, -6.096, -6.936]) #mV

i3 = np.array([0, -1.001, -2.011, -3.029, -4.043, -4.983, -5.987, -6.999, -8.037, 1.002, 2.02, 2.995, 4.043, 5.003, 6.045, 7.012, 7.993]) #mA
VH3 = np.array([-0.009, -1.227, -2.452, -3.689, -4.919, -6.06, -7.279, -8.506, -9.765, 1.21, 2.445, 3.629, 4.899, 6.065, 7.329, 8.501, 9.691]) #mV

i4 = np.array([0, 1.025, 2.002, 3.038, 4.058, 5.032, 6.012, 7, 8.006, -1.002, -2.002, -2.998, -4.034, -4.992, -6.013, -7.015, -7.987]) #mA
VH4 = np.array([-0.008, 1.592, 3.113, 4.728, 6.317, 7.833, 9.358, 10.896, 12.461, -1.57, -3.127, -4.678, -6.291, -7.781, -9.368, -10.926, -12.436]) #mV

i5 = np.array([0, -1.002, -2.002, -3.006, -3.992, -5.02, -6.003, -7.036, -8.021, 1.008, 1.988, 3, 3.997, 5.015, 5.962, 7.018, 8]) #mA
VH5 = np.array([-0.007, -1.891, -3.768, -5.653, -7.5, -9.415, -11.255, -13.189, -15.031, 1.884, 3.719, 5.614, 7.477, 9.381, 11.151, 13.127, 14.96]) #mV

hallVoltagePlotter = fitPlotter("HallVoltageVsCurrentBneg")

param1 = hallVoltagePlotter.addGraph(i1,VH1,title="i magnet -0.2 A")
param2 = hallVoltagePlotter.addGraph(i2,VH2,title="i magnet -0.4 A")
param3 = hallVoltagePlotter.addGraph(i3,VH3,title="i magnet -0.6 A")
param4 = hallVoltagePlotter.addGraph(i4,VH4,title="i magnet -0.8 A")
param5 = hallVoltagePlotter.addGraph(i5,VH5,title="i magnet -1.0 A")

hallVoltagePlotter.drawCanvas()
hallVoltagePlotter.saveCanvas("outputs/hallVoltageVsCurrentBneg.png")


--- fit Results for: i magnet -0.2 A ---
Function: pol1
Chi2/NDF: 0.0001 / 15
p-value:  1.0000

p0: -0.0071 +/- 0.0007
p1: 0.4682 +/- 0.0001
--------------------------------

--- fit Results for: i magnet -0.4 A ---
Function: pol1
Chi2/NDF: 0.0000 / 15
p-value:  1.0000

p0: -0.0085 +/- 0.0003
p1: 0.8657 +/- 0.0001
--------------------------------

--- fit Results for: i magnet -0.6 A ---
Function: pol1
Chi2/NDF: 0.0001 / 15
p-value:  1.0000

p0: -0.0097 +/- 0.0005
p1: 1.2140 +/- 0.0001
--------------------------------

--- fit Results for: i magnet -0.8 A ---
Function: pol1
Chi2/NDF: 0.0002 / 15
p-value:  1.0000

p0: -0.0053 +/- 0.0008
p1: 1.5572 +/- 0.0002
--------------------------------

--- fit Results for: i magnet -1.0 A ---
Function: pol1
Chi2/NDF: 0.0007 / 15
p-value:  1.0000

p0: -0.0129 +/- 0.0016
p1: 1.8729 +/- 0.0003
--------------------------------


python ERROR: cannot open image file "outputs/hallVoltageVsCurrentBneg.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/hallVoltageVsCurrentBneg.png


In [ ]:
# same mesaures but we inverted the magnetic field
# the error on i should be ( 0.5% rdg + 10 dgts ) what about the error of Vh?
# remember to implement errors

i1 = np.array([0, 1.007, 1.988, 2.991, 4.011, 5, 5.988, 7.009, 8.001, -1.002, -1.999, -3.006, -3.997, -5.039, -6.014, -6.953, -8.07])
VH1 = np.array([-0.002, -0.267, -0.525, -0.788, -1.056, -1.316, -1.575, -1.843, -2.104, 0.261, 0.522, 0.786, 1.046, 1.32, 1.576, 1.822, 2.115])

i2 = np.array([0, -1.025, -1.902, -2.997, -4.049, -4.955, -6.075, -7.068, -7.974, 0.922, 2.003, 3.02, 4.006, 5.021, 5.953, 6.998, 7.978])
VH2 = np.array([-0.002, 0.684, 1.267, 2.001, 2.704, 3.309, 4.057, 4.719, 5.325, -0.618, -1.345, -2.027, -2.688, -3.369, -3.995, -4.696, -5.355])

i3 = np.array([0, 1.011, 1.934, 2.984, 3.94, 4.944, 5.988, 7.042, 8.029, -0.985, -1.912, -3.007, -4.036, -4.945, -5.925, -6.98, -7.992])
VH3 = np.array([-0.001, -1.056, -2.021, -3.114, -4.112, -5.159, -6.248, -7.347, -8.377, 1.028, 1.997, 3.139, 4.211, 5.16, 6.183, 7.283, 8.338])

i4 = np.array([0, -1.006, -2.016, -2.972, -3.973, -5.01, -5.966, -6.996, -7.97, 0.975, 1.993, 2.997, 4.001, 4.983, 6.004, 6.978, 7.984])
VH4 = np.array([0, 1.419, 2.838, 4.186, 5.594, 7.053, 8.398, 9.846, 11.216, -1.372, -2.806, -4.217, -5.268, -7.011, -8.446, -9.814, -11.227])

i5 = np.array([0, 0.97, 1.998, 2.983, 4.002, 5.001, 6.009, 6.997, 7.971, -0.997, -2.007, -2.996, -3.987, -5.011, -6, -7.007, -8.013])
VH5 = np.array([0, -1.665, -3.415, -5.122, -6.867, -8.58, -10.308, -12.001, -13.668, 1.713, 3.443, 5.139, 6.847, 8.586, 10.28, 12.003, 13.728])

hallVoltagePlotter = fitPlotter("HallVoltageVsCurrentBpos")

param1 = hallVoltagePlotter.addGraph(i1,VH1,title="i magnet 0.2 A")
param2 = hallVoltagePlotter.addGraph(i2,VH2,title="i magnet 0.4 A")
param3 = hallVoltagePlotter.addGraph(i3,VH3,title="i magnet 0.6 A")
param4 = hallVoltagePlotter.addGraph(i4,VH4,title="i magnet 0.8 A")
param5 = hallVoltagePlotter.addGraph(i5,VH5,title="i magnet 1.0 A")

hallVoltagePlotter.drawCanvas()
hallVoltagePlotter.saveCanvas("outputs/hallVoltageVsCurrentBpos.png")

In [ ]:
BHallpos    = [1,2,3,4,5]
errBHallpos = [0.1]*5
BHallneg    = [-1,-2,-3,-4,-5]
errBHallneg = [0.1]*5

IHall1    = [2,3,6,8,12,16,20,24,28,32] 
errIHall1 = [0.1]*10
VHall1    = [1,2,3,4,5,6,7,8,9,10]
errVHall1 = [0.3]*10

# we have to remove the contribution for B=0
# (our measurements need to have the same currents, or else we need to interpolate between the points...)
def removeBackground(data, back, err_data, err_back):
    """ removes background data `back` from `data` (propagating errors as sums of squares) """
    newdata     = []
    err_newdata = []

    for i in range(0,len(data)):
        newdata.append(data[i] - back[i])
        err_newdata.append((err_data[i]**2 + err_back[i]**2)**0.5)

    return newdata, err_newdata

cleanIHall1, errcleanIHall1 = removeBackground(IHall1,Ip0,errIHall1,errIp0)

newHallCurrPlotter = fitPlotter("HallTensionVsCurrent")
param1 = newHallCurrPlotter.addGraph(IHall1, VHall1, errIHall1, errVHall1, "w/ background")
param2 = newHallCurrPlotter.addGraph(cleanIHall1, VHall1, errcleanIHall1, errVHall1, "w/out background")
newHallCurrPlotter.drawCanvas()
newHallCurrPlotter.saveCanvas("outputs/fit1.png")

# posso verificare che i parametri siano compatibili, e poi estrarre il valore di resistenza di hall
Zscore(param1,param2)

param = MeanError(param1,param2)

## Tensione di Hall al variare di $B$ (TODO)

Ripetiamo sostanzialmente le misure al punto precedente, ma invertendo i ruoli di variabile dipendente e indipendente. Adesso fissiamo $i_p$, scegliendo $3 \times 2$ valori, (dopo aver tracciato un'altra curva di caduta di potenziale ohmica a $B=0$) e studiamo il variare di $V_H$ con $B$.

**probabilmente ha senso riprendere i valori della curva ohmica (disallineamento) fissando i valori della corrente in base a questa nuova scansione: altrimenti dobbiamo interpolare lungo la curva di un fit**

In [24]:
# WARNING the arrays B1,B2 etc... contain current measurements!!! 
# same thing as before but now we are changing the magnetic field
# remember to implement the errors

B1 = np.array([-1.195, -0.88, -0.602, -0.287, 0, 0.41, 0.65, 0.902, 1.178])
VH1 = np.array([4.319, 3.483, 2.554, 1.371, 0.236, -1.376, -2.276, -3.148, -4.014])

B2 = np.array([1.177, 0.711, 0.552, 0.304, 0, -0.315, -0.597, -0.978, -1.238])
VH2 = np.array([-7.997, -5.371, -4.271, -2.413, -0.018, 2.468, 4.605, 7.252, 8.844])

B3 = np.array([-1.237, -0.894, -0.609, -0.334, 0, 0.342, 0.587, 0.915, 1.189])
VH3 = np.array([13.258, 10.537, 7.704, 4.642, 0.705, -3.335, -6.106, -9.544, -12.09])

hallVoltageBFieldPlotter = fitPlotter("HallVoltageVsMagneticFieldIpos")

param1 = hallVoltageBFieldPlotter.addGraph(B1,VH1,title="i sensor 0.2 A")
param2 = hallVoltageBFieldPlotter.addGraph(B2,VH2,title="i sensor 0.4 A")
param3 = hallVoltageBFieldPlotter.addGraph(B3,VH3,title="i sensor 0.6 A")

hallVoltageBFieldPlotter.drawCanvas()
hallVoltageBFieldPlotter.saveCanvas("outputs/hallVoltageVsBFieldIpos.png")


--- fit Results for: i sensor 0.2 A ---
Function: pol1
Chi2/NDF: 0.1321 / 7
p-value:  1.0000

p0: 0.1989 +/- 0.0458
p1: -3.6410 +/- 0.0589
--------------------------------

--- fit Results for: i sensor 0.4 A ---
Function: pol1
Chi2/NDF: 0.5929 / 7
p-value:  0.9990

p0: 0.0355 +/- 0.0972
p1: -7.2384 +/- 0.1274
--------------------------------

--- fit Results for: i sensor 0.6 A ---
Function: pol1
Chi2/NDF: 1.3249 / 7
p-value:  0.9878

p0: 0.5919 +/- 0.1450
p1: -10.8377 +/- 0.1851
--------------------------------


python ERROR: cannot open image file "outputs/hallVoltageVsBFieldIpos.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/hallVoltageVsBFieldIpos.png


In [27]:
# WARNING the arrays B1,B2 etc... contain current measurements!!! 
# now the current flows the other way
# remember to implement the errors


B1 = np.array([1.207, 0.9, 0.546, 0.277, 0, -0.3, -0.497, -0.918, -1.211])
VH1 = np.array([4.093, 3.307, 2.132, 1.118, 0.025, -1.16, -1.918, -3.426, -4.348])

B2 = np.array([-1.211, -0.907, -0.602, -0.312, 0, 0.303, 0.725, 0.897, 1.225])
VH2 = np.array([-8.697, -7.102, -5.088, -2.919, -0.462, 1.935, 5.078, 6.252, 8.284])

B3 = np.array([1.221, 0.879, 0.604, 0.321, 0, -0.4, -0.638, -0.928, -1.197])
VH3 = np.array([12.403, 9.723, 6.992, 3.838, 0.048, -4.667, -7.339, -10.372, -12.901])

hallVoltageBFieldPlotter = fitPlotter("HallVoltageVsMagneticFieldNeg")

param1 = hallVoltageBFieldPlotter.addGraph(B1,VH1,title="i sensor -0.2 A")
param2 = hallVoltageBFieldPlotter.addGraph(B2,VH2,title="i sensor -0.4 A")
param3 = hallVoltageBFieldPlotter.addGraph(B3,VH3,title="i sensor -0.6 A")

hallVoltageBFieldPlotter.drawCanvas()
hallVoltageBFieldPlotter.saveCanvas("outputs/hallVoltageVsBFieldNeg.png")


--- fit Results for: i sensor -0.2 A ---
Function: pol1
Chi2/NDF: 0.1439 / 7
p-value:  1.0000

p0: -0.0213 +/- 0.0478
p1: 3.6124 +/- 0.0624
--------------------------------

--- fit Results for: i sensor -0.4 A ---
Function: pol1
Chi2/NDF: 0.5510 / 7
p-value:  0.9992

p0: -0.3969 +/- 0.0935
p1: 7.2322 +/- 0.1178
--------------------------------

--- fit Results for: i sensor -0.6 A ---
Function: pol1
Chi2/NDF: 1.3809 / 7
p-value:  0.9861

p0: -0.0862 +/- 0.1481
p1: 10.8631 +/- 0.1879
--------------------------------


Warning in <TCanvas::Constructor>: Deleting canvas with same name: HallVoltageVsMagneticFieldNeg
python ERROR: cannot open image file "outputs/hallVoltageVsBFieldNeg.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/hallVoltageVsBFieldNeg.png


## Mobilità dei portatori

Misurando la caratteristica $I(V)$ del materiale seminconduttore (facendo variare la corrente $I$ tra -8mA e +8mA) che abbiamo usato nell'esperienza possiamo dare una stima della mobilità dei portatori di carica al suo interno. In particolare, dalla pendenza di $I(V)$ abbiamo la resistenza $R$, e quindi anche la resistività:
$$\rho =\frac{t\cdot w}{L} R$$
dove $t$ è lo spessore, $w$ la larghezza e $L$ la lunghezza.
Infine, dalla resistività si ha direttamente:
$$\mu = \frac{R_H}{\rho} (=R_H \sigma)$$
dove per $R_H$ possiamo considerare le stime date prima.

In [33]:
# fitting the curve I(V) as we need to find R in order tu calculate mu
# implement the errors

I1    = [0.537, 1.018, 1.514, 1.987, 2.499, 2.996, 3.498, 4.01, 4.49, 5.016, 5.451, 5.973, 6.47, 7, 7.5, 8]
errI1 = [0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001]
V1    = [34.804, 65.991, 98.036, 128.64, 161.732, 193.9, 226.29, 259.353, 290.42, 324.444, 352.526, 386.29, 418.42, 452.78, 485.152, 517.532]
errV1 = [0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001]

# do we need to make two plots or we just merge them together ?

I2    = [-0.501, -1.01, -1.503, -1.999, -2.492, -3, -3.499, -4.004, -4.494, -5.002, -5.504, -6.018, -6.5, -6.998, -7.505, -8.002]
errI2 = [0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001]
V2    = [-32.505, -64.937, -97.343, -129.5, -161.399, -194.292, -226.477, -259.082, -290.841, -323.673, -356.178, -389.277, -420.475, -452.676, -484.506, -517.489]
errV2 = [0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001,0.001]


muPlotter = fitPlotter("IVcharacteristic")
param = muPlotter.addGraph(I, V, errI, errV, title="I(V)")
muPlotter.drawCanvas()
muPlotter.saveCanvas("outputs/IVcharacteristic.png")

R, errR = param[1][0],param[1][1]


--- fit Results for: I(V) ---
Function: pol1
Chi2/NDF: 10.4829 / 14
p-value:  0.7261

p0: 0.1125 +/- 0.0340
p1: 64.6641 +/- 0.0070
--------------------------------


Warning in <TCanvas::Constructor>: Deleting canvas with same name: IVcharacteristic
python ERROR: cannot open image file "outputs/IVcharacteristic.png" for writing. Please check permissions.
Error in <TASImage::WriteImage>: error writing file outputs/IVcharacteristic.png


In [ ]:
# i think we need to measure the magnet's dimensions (?)

t = 1
w = 1
L = 1

rho = t * w * R / L

R_H = 1

mu = R_H / rho

## facoltativo: Misura del coefficiente di Hall a temperatura variabile

La misura della temperatura è effettuata mediante una misura di resistenza su un elemento
Pt100 (R$\simeq$ 100 $\ohm$). la tabella di calibrazione è su moodle.

Impostare la corrente iniettata nel campione a 8 mA e immergere il campione stesso nel
traferro, impostando un campo magnetico $\leq$ 200 mT.

A questo punto misuriamo la tensione di Hall $V_H$ (sottraendo il valore di riferimento in assenza di $B$) in funzione di $T$.

In [ ]:
RPt = [100,120,140,160,180] # fore non necessario
T_inc = np.arange(25,140,5)
T_dec = np.arange(140, 25, -5)

VH = np.arange(23) + np.random.rand(23)*3
VH_reference = np.random.rand(23)*3
print("\nT_inc: \n")
print(T_inc)
print("\nT_dec: \n")
print(T_dec)
print("\nV_H: \n")
print(VH)
print("\nV_H_reference: \n")
print(VH_reference)

In [ ]:
VH_inc_final = VH - VH_reference
VH_dec_final = VH - VH_reference

errVH = [0.3]*23
errT = [0.3]*23

In [ ]:
hallExtraIncPlotter = fitPlotter("VH_T_inc_characteristic")
VH_Tparam = hallExtraIncPlotter.addGraph(VH_inc_final, T_inc, errVH, errT, title="V_H (T) increasing")
hallExtraIncPlotter.drawCanvas()
hallExtraIncPlotter.saveCanvas("outputs/VH_T_inc_characteristic.png")

hallExtraDecPlotter = fitPlotter("VH_T_dec_characteristic")
VH_Tparam = hallExtraDecPlotter.addGraph(VH_dec_final, T_dec, errVH, errT, title="V_H (T) decreasing")
hallExtraDecPlotter.drawCanvas()
hallExtraDecPlotter.saveCanvas("outputs/VH_T_dec_characteristic.png")